# PS S6E6 - 033 Blend add032 TabM check

Experiment: `exp_20260609_033_blend_add032_tabm_check`

Purpose:
- Add `032 one-vs-rest TabM as-is` to the current candidate pool.
- Check whether 032 adds useful low-correlation / error-complement signal to the 031 current best blend and its core components.
- Keep this as a diagnostic before moving to 034 GPU logistic regression stacker.

Current context:
- 031 best: `W_018_019_027_026_028_030 / hc_nonnegative_raw`
  - CV: `0.9700333620193362`
  - Public LB: `0.97043`
- 032 TabM:
  - RAW CV: `0.9610105651284856`
  - tuned CV: `0.9686168281485955`
  - RAW Public LB: `0.96106`
  - biased Public LB: `0.96895`
  - role: TabM / alternate NN-family auxiliary material.
- Important implementation note:
  - The 032 `.npy` files are probability material for blending. If they are raw normalized probabilities, the recomputed OOF score will be close to RAW CV, not the biased submission CV.

Important:
- `LogReg / Ridge / ElasticNet` rows are diagnostic only. Final candidates should come from safe methods (`avg`, `hc_nonnegative_raw`, `nm_softmax_raw`) unless a fold-safe nested stacker is built.
- Do not judge 032 only by standalone LB. The key question is whether it receives non-zero safe blend weight and improves CV.
- If 032 does not improve the safe blend or weight is ~0, move on to 034.


In [1]:
# ============================================================
# 0. Imports / Config
# ============================================================

import json
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import spearmanr

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import LogisticRegression, RidgeClassifier, Ridge, ElasticNet

warnings.filterwarnings("ignore")

try:
    import yaml
except Exception:
    yaml = None

COMPETITION = "PS S6E6 Predicting Stellar Class"
EXP_ID = "exp_20260609_033_blend_add032_tabm_check"
SEED = 42

TRAIN_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/train.csv")
TEST_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/test.csv")
SAMPLE_SUB_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv")

# Primary location. The loader below also scans /kaggle/input if a file is not found here.
NPY_DIR = Path("/kaggle/input/datasets/mizushimatoshihiko/npy-files")
INPUT_ROOT = Path("/kaggle/input")

OUTDIR = Path("/kaggle/working") / EXP_ID
OUTDIR.mkdir(parents=True, exist_ok=True)

ID_COL = "id"
TARGET_COL = "class"

MODEL_SPECS = [
    {
        "key": "015_realmlp_seed42",
        "exp_id": "exp_20260605_015_shared001_realmlp_as_is",
        "family": "RealMLP",
        "role": "stable_single_realmlp_backup",
        "oof": "oof_exp_20260605_015_shared001_realmlp_as_is_proba.npy",
        "pred": "pred_exp_20260605_015_shared001_realmlp_as_is_proba.npy",
        "cv": 0.9681693449222352,
        "public_lb": 0.96977,
    },
    {
        "key": "017_realmlp_bias",
        "exp_id": "exp_20260606_017_bias_search_on_015_realmlp",
        "family": "RealMLP",
        "role": "previous_best_realmlp_bias_backup",
        "oof": "oof_exp_20260606_017_bias_search_on_015_realmlp_proba.npy",
        "pred": "pred_exp_20260606_017_bias_search_on_015_realmlp_proba.npy",
        "cv": 0.9683022653955233,
        "public_lb": 0.96985,
    },
    {
        "key": "018_xgb_shared003",
        "exp_id": "exp_20260606_018_shared003_xgb_as_is",
        "family": "XGBoost",
        "role": "effective_blend_material",
        "oof": "oof_exp_20260606_018_shared003_xgb_as_is_proba.npy",
        "pred": "pred_exp_20260606_018_shared003_xgb_as_is_proba.npy",
        "cv": 0.965207986418628,
        "public_lb": 0.96578,
    },
    {
        "key": "019_blend_best",
        "exp_id": "exp_20260607_019_blend_add018_xgb_check",
        "family": "Blend",
        "role": "previous_public_confirmed_best",
        "oof": "oof_exp_20260607_019_blend_add018_xgb_check_best_blend_proba.npy",
        "pred": "pred_exp_20260607_019_blend_add018_xgb_check_best_blend_proba.npy",
        "cv": 0.968872315017554,
        "public_lb": 0.97000,
    },
    {
        "key": "020_blend_bias",
        "exp_id": "exp_20260607_020_bias_search_on_019_best_blend",
        "family": "Blend",
        "role": "cv_trusted_attack_reference",
        "oof": "oof_exp_20260607_020_bias_search_on_019_best_blend_proba.npy",
        "pred": "pred_exp_20260607_020_bias_search_on_019_best_blend_proba.npy",
        "cv": 0.9692240443617589,
        "public_lb": 0.96969,
    },
    {
        "key": "024_seed_ensemble_blend",
        "exp_id": "exp_20260608_024_seed_ensemble_and_blend_check",
        "family": "Blend",
        "role": "seed_ensemble_blend_reference",
        "oof": "oof_exp_20260608_024_seed_ensemble_and_blend_check_best_blend_proba.npy",
        "pred": "pred_exp_20260608_024_seed_ensemble_and_blend_check_best_blend_proba.npy",
        "cv": 0.9694803109983217,
        "public_lb": 0.96958,
    },
    {
        "key": "026_realmlp_v5",
        "exp_id": "exp_20260608_026_realmlp_v5_as_is",
        "family": "RealMLP",
        "role": "realmlp_v5_single_model_candidate",
        "oof": "oof_exp_20260608_026_realmlp_v5_as_is_proba.npy",
        "pred": "pred_exp_20260608_026_realmlp_v5_as_is_proba.npy",
        "cv": 0.9690389777981231,
        "public_lb": 0.96979,
    },
    {
        "key": "027_blend_add026",
        "exp_id": "exp_20260608_027_blend_add026_realmlp_v5_check",
        "family": "Blend",
        "role": "previous_cv_trusted_slot",
        "oof": "oof_exp_20260608_027_blend_add026_realmlp_v5_check_best_blend_proba.npy",
        "pred": "pred_exp_20260608_027_blend_add026_realmlp_v5_check_best_blend_proba.npy",
        "cv": 0.9695425457477898,
        "public_lb": 0.96975,
    },
    {
        "key": "028_cat_v3",
        "exp_id": "exp_20260608_028_cat_v3_as_is",
        "family": "CatBoost",
        "role": "catboost_v3_family_aux_material",
        "oof": "oof_exp_20260608_028_cat_v3_as_is_proba.npy",
        "pred": "pred_exp_20260608_028_cat_v3_as_is_proba.npy",
        "cv": 0.9688146470512534,
        "public_lb": 0.96969,
    },
    {
        "key": "029_blend_add028",
        "exp_id": "exp_20260608_029_blend_add028_cat_v3_check",
        "family": "Blend",
        "role": "previous_best_backup",
        "oof": "oof_exp_20260608_029_blend_add028_cat_v3_check_best_blend_proba.npy",
        "pred": "pred_exp_20260608_029_blend_add028_cat_v3_check_best_blend_proba.npy",
        "cv": 0.9700023026377228,
        "public_lb": 0.970036,
    },
    {
        "key": "030_ovr_xgb",
        "exp_id": "exp_20260608_030_ovr_xgb_as_is",
        "family": "XGBoost",
        "role": "low_weight_auxiliary_xgb_ovr_material",
        "oof": "oof_exp_20260608_030_ovr_xgb_as_is_proba.npy",
        "pred": "pred_exp_20260608_030_ovr_xgb_as_is_proba.npy",
        "cv": 0.9609971296999271,
        "public_lb": 0.96118,
    },
    {
        "key": "031_blend_add030",
        "exp_id": "exp_20260608_031_blend_add030_ovr_xgb_check",
        "family": "Blend",
        "role": "public_confirmed_current_best",
        "oof": "oof_exp_20260608_031_blend_add030_ovr_xgb_check_best_blend_proba.npy",
        "pred": "pred_exp_20260608_031_blend_add030_ovr_xgb_check_best_blend_proba.npy",
        "cv": 0.9700333620193362,
        "public_lb": 0.97043,
    },
    {
        "key": "032_ovr_tabm",
        "exp_id": "exp_20260609_032_ovr_tabm_as_is",
        "family": "TabM",
        "role": "tabm_ovr_family_aux_material",
        "oof": "oof_exp_20260609_032_ovr_tabm_as_is_proba.npy",
        "pred": "pred_exp_20260609_032_ovr_tabm_as_is_proba.npy",
        # This should match the probability npy recomputed score if the npy is raw normalized probabilities.
        # The biased submission CV/LB are tracked separately below.
        "cv": 0.9610105651284856,
        "public_lb": 0.96106,
        "tuned_cv": 0.9686168281485955,
        "public_lb_biased": 0.96895,
    },
]

# 033 should answer one question first:
# Does 032 TabM add complementary signal to 031, or to the 027/026/028/030 core?
BLEND_SETS = {
    # Singles / current references
    "A_031_only": ["031_blend_add030"],
    "B_032_only": ["032_ovr_tabm"],
    "C_029_only": ["029_blend_add028"],
    "D_027_only": ["027_blend_add026"],
    "E_026_only": ["026_realmlp_v5"],
    "F_028_only": ["028_cat_v3"],
    "G_030_only": ["030_ovr_xgb"],
    "H_018_only": ["018_xgb_shared003"],

    # Direct add032 checks
    "I_031_032": ["031_blend_add030", "032_ovr_tabm"],
    "J_029_032": ["029_blend_add028", "032_ovr_tabm"],
    "K_027_032": ["027_blend_add026", "032_ovr_tabm"],
    "L_026_032": ["026_realmlp_v5", "032_ovr_tabm"],
    "M_028_032": ["028_cat_v3", "032_ovr_tabm"],
    "N_030_032": ["030_ovr_xgb", "032_ovr_tabm"],
    "O_018_032": ["018_xgb_shared003", "032_ovr_tabm"],

    # Current best and core components with 032
    "P_031_029_032": ["031_blend_add030", "029_blend_add028", "032_ovr_tabm"],
    "Q_031_028_032": ["031_blend_add030", "028_cat_v3", "032_ovr_tabm"],
    "R_031_026_032": ["031_blend_add030", "026_realmlp_v5", "032_ovr_tabm"],
    "S_031_030_032": ["031_blend_add030", "030_ovr_xgb", "032_ovr_tabm"],
    "T_029_027_026_028_030_032": ["029_blend_add028", "027_blend_add026", "026_realmlp_v5", "028_cat_v3", "030_ovr_xgb", "032_ovr_tabm"],
    "U_027_026_028_030_032": ["027_blend_add026", "026_realmlp_v5", "028_cat_v3", "030_ovr_xgb", "032_ovr_tabm"],
    "V_031_027_026_028_030_032": ["031_blend_add030", "027_blend_add026", "026_realmlp_v5", "028_cat_v3", "030_ovr_xgb", "032_ovr_tabm"],

    # Rebuild 031 underlying component pool plus 032.
    "W_018_019_027_026_028_030_032": ["018_xgb_shared003", "019_blend_best", "027_blend_add026", "026_realmlp_v5", "028_cat_v3", "030_ovr_xgb", "032_ovr_tabm"],
    "X_018_019_027_026_028_032": ["018_xgb_shared003", "019_blend_best", "027_blend_add026", "026_realmlp_v5", "028_cat_v3", "032_ovr_tabm"],

    # Broader diagnostic pools. Interpret weights carefully because composite blends and components coexist.
    "Y_031_029_027_026_028_030_032": ["031_blend_add030", "029_blend_add028", "027_blend_add026", "026_realmlp_v5", "028_cat_v3", "030_ovr_xgb", "032_ovr_tabm"],
    "Z_full_015_017_018_019_020_024_026_027_028_029_030_031_032": [
        "015_realmlp_seed42", "017_realmlp_bias", "018_xgb_shared003",
        "019_blend_best", "020_blend_bias", "024_seed_ensemble_blend",
        "026_realmlp_v5", "027_blend_add026", "028_cat_v3",
        "029_blend_add028", "030_ovr_xgb", "031_blend_add030", "032_ovr_tabm",
    ],
}

print("EXP_ID:", EXP_ID)
print("OUTDIR:", OUTDIR)
print("NPY_DIR:", NPY_DIR)
print("n_models:", len(MODEL_SPECS))
print("n_blend_sets:", len(BLEND_SETS))


EXP_ID: exp_20260609_033_blend_add032_tabm_check
OUTDIR: /kaggle/working/exp_20260609_033_blend_add032_tabm_check
NPY_DIR: /kaggle/input/datasets/mizushimatoshihiko/npy-files
n_models: 13
n_blend_sets: 26


In [2]:
# ============================================================
# 1. Utilities
# ============================================================

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)

def resolve_npy(filename):
    """Resolve an npy path. Prefer NPY_DIR, then scan /kaggle/input.

    This is intentionally more robust than 027 because 028 may be uploaded as a
    separate dataset from older npy files.
    """
    p = NPY_DIR / filename
    if p.exists():
        return p
    matches = list(INPUT_ROOT.rglob(filename)) if INPUT_ROOT.exists() else []
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        print(f"[WARN] multiple matches for {filename}. Using first:")
        for m in matches[:10]:
            print("  ", m)
        return matches[0]
    raise FileNotFoundError(f"Missing npy file: {filename}. Checked {NPY_DIR} and scanned {INPUT_ROOT}.")

def validate_proba(name, arr, n_rows, n_classes):
    assert arr.shape == (n_rows, n_classes), (name, arr.shape)
    assert np.isfinite(arr).all(), name
    assert np.allclose(arr.sum(axis=1), 1.0, atol=1e-4), name
    assert arr.min() >= -1e-7, (name, arr.min())
    assert arr.max() <= 1.0 + 1e-7, (name, arr.max())

def normalize_rows(p, eps=1e-12):
    p = np.asarray(p, dtype=np.float64)
    p = np.clip(p, eps, None)
    return p / p.sum(axis=1, keepdims=True)

def softmax_np(x, axis=1):
    x = np.asarray(x, dtype=np.float64)
    x = x - np.max(x, axis=axis, keepdims=True)
    ex = np.exp(x)
    return ex / np.sum(ex, axis=axis, keepdims=True)

def proba_to_pred(p):
    return np.argmax(p, axis=1)

def flatten_proba(p):
    return np.asarray(p, dtype=float).reshape(-1)

def corr_pearson(a, b):
    a, b = flatten_proba(a), flatten_proba(b)
    if np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])

def corr_spearman(a, b):
    a, b = flatten_proba(a), flatten_proba(b)
    if np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(spearmanr(a, b).correlation)

def class_recalls(y_true, y_pred, class_names):
    out = {}
    for i, cls in enumerate(class_names):
        m = y_true == i
        out[f"recall_{cls}"] = float((y_pred[m] == i).mean()) if m.any() else float("nan")
    return out

def evaluate_proba(y_true, p, class_names):
    pred = proba_to_pred(p)
    res = {"balanced_accuracy": float(balanced_accuracy_score(y_true, pred))}
    res.update(class_recalls(y_true, pred, class_names))
    return res

def rank_normalize_matrix(p):
    out = np.zeros_like(p, dtype=np.float64)
    n = p.shape[0]
    for j in range(p.shape[1]):
        r = pd.Series(p[:, j]).rank(method="average").to_numpy()
        out[:, j] = (r - 1) / max(n - 1, 1)
    return normalize_rows(out)

def logit_transform(p, eps=1e-15):
    p = np.clip(p, eps, 1.0 - eps)
    return np.log(p / (1.0 - p))

def build_meta_features(keys, proba_dict, mode):
    mats = []
    for k in keys:
        p = proba_dict[k]
        if mode == "raw":
            mats.append(p)
        elif mode == "rank":
            mats.append(rank_normalize_matrix(p))
        elif mode == "logit":
            mats.append(logit_transform(p))
        elif mode == "raw_logit":
            mats += [p, logit_transform(p)]
        elif mode == "raw_rank_logit":
            mats += [p, rank_normalize_matrix(p), logit_transform(p)]
        else:
            raise ValueError(mode)
    return np.concatenate(mats, axis=1)

def average_proba(keys, oof_dict, pred_dict=None):
    oof_avg = normalize_rows(np.mean([oof_dict[k] for k in keys], axis=0))
    pred_avg = None
    if pred_dict is not None:
        pred_avg = normalize_rows(np.mean([pred_dict[k] for k in keys], axis=0))
    return oof_avg, pred_avg

def onehot_y(y, n_classes):
    out = np.zeros((len(y), n_classes), dtype=np.float64)
    out[np.arange(len(y)), y] = 1.0
    return out

def hill_climb_weights(y_true, probas, steps=(0.1, 0.05, 0.02, 0.01, 0.005, 0.002), max_iter=300):
    n = len(probas)
    w = np.ones(n, dtype=float) / n
    cur_score = balanced_accuracy_score(y_true, normalize_rows(sum(w[i] * probas[i] for i in range(n))).argmax(axis=1))
    hist = [{"iter": 0, "score": float(cur_score), "weights": w.copy().tolist()}]
    for step in steps:
        improved = True
        it = 0
        while improved and it < max_iter:
            improved = False
            it += 1
            best_score, best_w = cur_score, w.copy()
            for i in range(n):
                for direction in [-1, 1]:
                    cand_w = w.copy()
                    cand_w[i] += direction * step
                    cand_w = np.clip(cand_w, 0.0, None)
                    if cand_w.sum() <= 0:
                        continue
                    cand_w /= cand_w.sum()
                    cand = normalize_rows(sum(cand_w[j] * probas[j] for j in range(n)))
                    score = balanced_accuracy_score(y_true, cand.argmax(axis=1))
                    if score > best_score + 1e-15:
                        best_score, best_w = score, cand_w.copy()
            if best_score > cur_score + 1e-15:
                cur_score, w = best_score, best_w
                improved = True
                hist.append({"iter": len(hist), "step": step, "score": float(cur_score), "weights": w.copy().tolist()})
    return w, float(cur_score), hist

def nm_softmax_weights(y_true, probas, n_restarts=5, maxiter=2500):
    probas = [np.asarray(p, dtype=np.float64) for p in probas]
    n = len(probas)

    def eval_from_theta(theta):
        w = np.exp(theta - np.max(theta))
        w = w / w.sum()
        p = normalize_rows(sum(w[i] * probas[i] for i in range(n)))
        return balanced_accuracy_score(y_true, p.argmax(axis=1)), w

    def objective(theta):
        score, _ = eval_from_theta(theta)
        return -score

    inits = [np.zeros(n)]
    rng = np.random.default_rng(SEED)
    for _ in range(n_restarts - 1):
        inits.append(rng.normal(0, 0.25, size=n))

    best_score, best_w, best_res = -1, None, None
    for init in inits:
        res = minimize(objective, init, method="Nelder-Mead", options={"maxiter": maxiter, "xatol": 1e-8, "fatol": 1e-12})
        score, w = eval_from_theta(res.x)
        if score > best_score:
            best_score, best_w, best_res = score, w, res
    return best_w, float(best_score), best_res

def disagreement_and_error_overlap(y_true, p1, p2):
    pred1 = p1.argmax(axis=1)
    pred2 = p2.argmax(axis=1)
    wrong1 = pred1 != y_true
    wrong2 = pred2 != y_true
    both_wrong = wrong1 & wrong2
    either_wrong = wrong1 | wrong2
    return {
        "disagreement_rate": float((pred1 != pred2).mean()),
        "wrong_rate_left": float(wrong1.mean()),
        "wrong_rate_right": float(wrong2.mean()),
        "both_wrong_rate": float(both_wrong.mean()),
        "error_overlap_jaccard": float(both_wrong.sum() / max(either_wrong.sum(), 1)),
        "left_wrong_right_correct_rate": float((wrong1 & ~wrong2).mean()),
        "right_wrong_left_correct_rate": float((wrong2 & ~wrong1).mean()),
    }


In [3]:
# ============================================================
# 2. Load data and OOF/pred
# ============================================================

for p in [TRAIN_PATH, TEST_PATH, SAMPLE_SUB_PATH]:
    if not p.exists():
        raise FileNotFoundError(p)

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_SUB_PATH)

le = LabelEncoder()
y = le.fit_transform(train[TARGET_COL].astype(str))
class_names = le.classes_.tolist()
n_classes = len(class_names)
assert class_names == ["GALAXY", "QSO", "STAR"], class_names

oof, pred, resolved_specs = {}, {}, []
for spec in MODEL_SPECS:
    key = spec["key"]
    oof_path = resolve_npy(spec["oof"])
    pred_path = resolve_npy(spec["pred"])
    oof_arr = np.load(oof_path)
    pred_arr = np.load(pred_path)
    validate_proba(f"oof:{key}", oof_arr, len(train), n_classes)
    validate_proba(f"pred:{key}", pred_arr, len(test), n_classes)
    oof[key] = oof_arr.astype(np.float64)
    pred[key] = pred_arr.astype(np.float64)
    s = spec.copy()
    s["resolved_oof_path"] = str(oof_path)
    s["resolved_pred_path"] = str(pred_path)
    resolved_specs.append(s)
    print(f"loaded {key}: {oof_arr.shape} / {pred_arr.shape}")

model_keys = [s["key"] for s in resolved_specs]
print("class_names:", class_names)
print("loaded keys:", model_keys)


loaded 015_realmlp_seed42: (577347, 3) / (247435, 3)
loaded 017_realmlp_bias: (577347, 3) / (247435, 3)
loaded 018_xgb_shared003: (577347, 3) / (247435, 3)
loaded 019_blend_best: (577347, 3) / (247435, 3)
loaded 020_blend_bias: (577347, 3) / (247435, 3)
loaded 024_seed_ensemble_blend: (577347, 3) / (247435, 3)
loaded 026_realmlp_v5: (577347, 3) / (247435, 3)
loaded 027_blend_add026: (577347, 3) / (247435, 3)
loaded 028_cat_v3: (577347, 3) / (247435, 3)
loaded 029_blend_add028: (577347, 3) / (247435, 3)
loaded 030_ovr_xgb: (577347, 3) / (247435, 3)
loaded 031_blend_add030: (577347, 3) / (247435, 3)
loaded 032_ovr_tabm: (577347, 3) / (247435, 3)
class_names: ['GALAXY', 'QSO', 'STAR']
loaded keys: ['015_realmlp_seed42', '017_realmlp_bias', '018_xgb_shared003', '019_blend_best', '020_blend_bias', '024_seed_ensemble_blend', '026_realmlp_v5', '027_blend_add026', '028_cat_v3', '029_blend_add028', '030_ovr_xgb', '031_blend_add030', '032_ovr_tabm']


In [4]:
# ============================================================
# 3. Individual scores and pairwise diagnostics
# ============================================================

rows = []
for spec in resolved_specs:
    key = spec["key"]
    p = oof[key]
    y_pred = proba_to_pred(p)
    score = balanced_accuracy_score(y, y_pred)
    row = {
        "key": key,
        "exp_id": spec["exp_id"],
        "family": spec["family"],
        "role": spec["role"],
        "cv_from_memo": spec.get("cv", np.nan),
        "public_lb": spec.get("public_lb", np.nan),
        "public_lb_biased": spec.get("public_lb_biased", np.nan),
        "tuned_cv": spec.get("tuned_cv", np.nan),
        "recomputed_oof_cv": float(score),
        "delta_recomputed_minus_memo": float(score - spec.get("cv", score)),
    }
    row.update(class_recalls(y, y_pred, class_names))
    rows.append(row)

individual_df = pd.DataFrame(rows).sort_values("recomputed_oof_cv", ascending=False).reset_index(drop=True)
display(individual_df)
individual_df.to_csv(OUTDIR / "individual_scores.csv", index=False)

pair_rows = []
for a, b in combinations(model_keys, 2):
    diag = disagreement_and_error_overlap(y, oof[a], oof[b])
    pair_rows.append({
        "left": a,
        "right": b,
        "pearson_flat_proba": corr_pearson(oof[a], oof[b]),
        "spearman_flat_proba": corr_spearman(oof[a], oof[b]),
        **diag,
    })

pairwise_df = pd.DataFrame(pair_rows)
pairwise_df = pairwise_df.sort_values(["pearson_flat_proba", "error_overlap_jaccard"], ascending=[True, True]).reset_index(drop=True)
display(pairwise_df)
pairwise_df.to_csv(OUTDIR / "pairwise_oof_correlation.csv", index=False)

# Focus table for 032.
focus_032_df = pairwise_df[(pairwise_df["left"] == "032_ovr_tabm") | (pairwise_df["right"] == "032_ovr_tabm")].copy()
focus_032_df = focus_032_df.sort_values("pearson_flat_proba").reset_index(drop=True)
display(focus_032_df)
focus_032_df.to_csv(OUTDIR / "pairwise_oof_correlation_focus_032.csv", index=False)

# Also keep compact focus tables for current best 031 and previous best 029.
focus_031_df = pairwise_df[(pairwise_df["left"] == "031_blend_add030") | (pairwise_df["right"] == "031_blend_add030")].copy()
focus_031_df = focus_031_df.sort_values("pearson_flat_proba").reset_index(drop=True)
display(focus_031_df)
focus_031_df.to_csv(OUTDIR / "pairwise_oof_correlation_focus_031.csv", index=False)

focus_029_df = pairwise_df[(pairwise_df["left"] == "029_blend_add028") | (pairwise_df["right"] == "029_blend_add028")].copy()
focus_029_df = focus_029_df.sort_values("pearson_flat_proba").reset_index(drop=True)
display(focus_029_df)
focus_029_df.to_csv(OUTDIR / "pairwise_oof_correlation_focus_029.csv", index=False)


,key,exp_id,family,role,cv_from_memo,public_lb,public_lb_biased,tuned_cv,recomputed_oof_cv,delta_recomputed_minus_memo,recall_GALAXY,recall_QSO,recall_STAR
0,031_blend_add030,exp_20260608_031_blend_add030_ovr_xgb_check,Blend,public_confirmed_current_best,0.970033,0.970430,NaN,NaN,0.970033,0.0,0.961738,0.975790,0.972571
1,029_blend_add028,exp_20260608_029_blend_add028_cat_v3_check,Blend,previous_best_backup,0.970002,0.970036,NaN,NaN,0.970002,0.0,0.961315,0.975867,0.972825
2,027_blend_add026,exp_20260608_027_blend_add026_realmlp_v5_check,Blend,previous_cv_trusted_slot,0.969543,0.969750,NaN,NaN,0.969543,0.0,0.961479,0.976439,0.970710
3,024_seed_ensemble_blend,exp_20260608_024_seed_ensemble_and_blend_check,Blend,seed_ensemble_blend_reference,0.969480,0.969580,NaN,NaN,0.969480,0.0,0.961956,0.976174,0.970311
4,020_blend_bias,exp_20260607_020_bias_search_on_019_best_blend,Blend,cv_trusted_attack_reference,0.969224,0.969690,NaN,NaN,0.969224,0.0,0.961137,0.976200,0.970335
5,026_realmlp_v5,exp_20260608_026_realmlp_v5_as_is,RealMLP,realmlp_v5_single_model_candidate,0.969039,0.969790,NaN,NaN,0.969039,0.0,0.959505,0.976431,0.971181
6,019_blend_best,exp_20260607_019_blend_add018_xgb_check,Blend,previous_public_confirmed_best,0.968872,0.970000,NaN,NaN,0.968872,0.0,0.965479,0.974441,0.966696
7,028_cat_v3,exp_20260608_028_cat_v3_as_is,CatBoost,catboost_v3_family_aux_material,0.968815,0.969690,NaN,NaN,0.968815,0.0,0.960099,0.974826,0.971520
8,017_realmlp_bias,exp_20260606_017_bias_search_on_015_realmlp,RealMLP,previous_best_realmlp_bias_backup,0.968302,0.969850,NaN,NaN,0.968302,0.0,0.959889,0.975867,0.969150
9,015_realmlp_seed42,exp_20260605_015_shared001_realmlp_as_is,RealMLP,stable_single_realmlp_backup,0.968169,0.969770,NaN,NaN,0.968169,0.0,0.962234,0.976098,0.966177


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,026_realmlp_v5,030_ovr_xgb,0.987581,0.751435,0.022543,0.035388,0.028955,0.021039,0.485861,0.014348,0.007916
1,026_realmlp_v5,032_ovr_tabm,0.988073,0.761653,0.022150,0.035388,0.029062,0.021273,0.492699,0.014115,0.007789
2,017_realmlp_bias,030_ovr_xgb,0.988127,0.889617,0.022371,0.035542,0.028955,0.021218,0.490255,0.014324,0.007737
3,017_realmlp_bias,032_ovr_tabm,0.988556,0.896884,0.022018,0.035542,0.029062,0.021439,0.496690,0.014102,0.007623
4,028_cat_v3,030_ovr_xgb,0.988941,0.965161,0.021862,0.035277,0.028955,0.021330,0.497194,0.013947,0.007625
...,...,...,...,...,...,...,...,...,...,...,...
73,020_blend_bias,024_seed_ensemble_blend,0.999654,0.974434,0.003634,0.034489,0.033962,0.032441,0.900914,0.002047,0.001521
74,019_blend_best,020_blend_bias,0.999766,0.997020,0.003752,0.032528,0.034489,0.031648,0.894809,0.000880,0.002841
75,024_seed_ensemble_blend,027_blend_add026,0.999876,0.925081,0.001959,0.033962,0.034163,0.033098,0.944914,0.000864,0.001065
76,015_realmlp_seed42,017_realmlp_bias,0.999895,0.997811,0.002175,0.034388,0.035542,0.033907,0.941244,0.000482,0.001635


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,026_realmlp_v5,032_ovr_tabm,0.988073,0.761653,0.022150,0.035388,0.029062,0.021273,0.492699,0.014115,0.007789
1,017_realmlp_bias,032_ovr_tabm,0.988556,0.896884,0.022018,0.035542,0.029062,0.021439,0.496690,0.014102,0.007623
2,028_cat_v3,032_ovr_tabm,0.989126,0.963762,0.021803,0.035277,0.029062,0.021407,0.498608,0.013870,0.007656
3,015_realmlp_seed42,032_ovr_tabm,0.989870,0.909912,0.020289,0.034388,0.029062,0.021724,0.520609,0.012665,0.007339
4,020_blend_bias,032_ovr_tabm,0.990438,0.916090,0.020653,0.034489,0.029062,0.021566,0.513655,0.012923,0.007496
5,029_blend_add028,032_ovr_tabm,0.990460,0.831713,0.020577,0.034083,0.029062,0.021396,0.512488,0.012687,0.007666
6,027_blend_add026,032_ovr_tabm,0.990561,0.838693,0.020395,0.034163,0.029062,0.021526,0.516220,0.012637,0.007536
7,018_xgb_shared003,032_ovr_tabm,0.990669,0.965070,0.018556,0.033536,0.029062,0.022139,0.547198,0.011397,0.006923
8,031_blend_add030,032_ovr_tabm,0.990777,0.830573,0.020262,0.033858,0.029062,0.021441,0.516912,0.012417,0.007621
9,024_seed_ensemble_blend,032_ovr_tabm,0.990953,0.927651,0.019976,0.033962,0.029062,0.021635,0.522723,0.012327,0.007427


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,030_ovr_xgb,031_blend_add030,0.990536,0.821836,0.020385,0.028955,0.033858,0.021341,0.514576,0.007614,0.012518
1,031_blend_add030,032_ovr_tabm,0.990777,0.830573,0.020262,0.033858,0.029062,0.021441,0.516912,0.012417,0.007621
2,018_xgb_shared003,031_blend_add030,0.993874,0.858577,0.014553,0.033536,0.033858,0.026518,0.648729,0.007018,0.007340
3,015_realmlp_seed42,031_blend_add030,0.997199,0.868061,0.009734,0.034388,0.033858,0.029364,0.755178,0.005025,0.004495
4,017_realmlp_bias,031_blend_add030,0.997253,0.881326,0.009913,0.035542,0.033858,0.029842,0.754368,0.005700,0.004017
5,019_blend_best,031_blend_add030,0.998293,0.878896,0.007571,0.032528,0.033858,0.029480,0.798761,0.003048,0.004379
6,020_blend_bias,031_blend_add030,0.998538,0.899859,0.007276,0.034489,0.033858,0.030600,0.810673,0.003888,0.003258
7,026_realmlp_v5,031_blend_add030,0.998540,0.976821,0.007079,0.035388,0.033858,0.031162,0.818219,0.004226,0.002697
8,024_seed_ensemble_blend,031_blend_add030,0.998741,0.900716,0.006753,0.033962,0.033858,0.030599,0.822057,0.003364,0.003260
9,028_cat_v3,031_blend_add030,0.998742,0.872885,0.006942,0.035277,0.033858,0.031167,0.820857,0.004110,0.002692


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,029_blend_add028,030_ovr_xgb,0.990184,0.822720,0.020717,0.034083,0.028955,0.021287,0.509853,0.012796,0.007668
1,029_blend_add028,032_ovr_tabm,0.990460,0.831713,0.020577,0.034083,0.029062,0.021396,0.512488,0.012687,0.007666
2,018_xgb_shared003,029_blend_add028,0.993841,0.859932,0.014596,0.033536,0.034083,0.026611,0.648927,0.006925,0.007472
3,015_realmlp_seed42,029_blend_add028,0.997290,0.870762,0.009627,0.034388,0.034083,0.029523,0.757994,0.004865,0.004561
4,017_realmlp_bias,029_blend_add028,0.997375,0.884010,0.009729,0.035542,0.034083,0.030039,0.758827,0.005503,0.004044
5,019_blend_best,029_blend_add028,0.998329,0.881267,0.007545,0.032528,0.034083,0.029604,0.799963,0.002924,0.004479
6,026_realmlp_v5,029_blend_add028,0.998564,0.976314,0.006989,0.035388,0.034083,0.031314,0.820654,0.004074,0.002770
7,020_blend_bias,029_blend_add028,0.998627,0.902219,0.007079,0.034489,0.034083,0.030808,0.815805,0.003681,0.003275
8,028_cat_v3,029_blend_add028,0.998697,0.874527,0.007069,0.035277,0.034083,0.031224,0.818739,0.004053,0.002860
9,024_seed_ensemble_blend,029_blend_add028,0.998828,0.903624,0.006568,0.033962,0.034083,0.030796,0.826746,0.003166,0.003287


In [5]:
# ============================================================
# 4. Blend diagnostics
# ============================================================

blend_rows = {}
blend_probas = {}

def _weight_of(keys, weights, target_key):
    if weights is None or target_key not in keys:
        return None
    return float(dict(zip(keys, weights)).get(target_key, 0.0))

def record_blend(set_name, method, keys, oof_p, test_p=None, weights=None, extra=None):
    ev = evaluate_proba(y, oof_p, class_names)
    row = {
        "set_name": set_name,
        "method": method,
        "keys": ",".join(keys),
        "n_models": len(keys),
        "balanced_accuracy": ev["balanced_accuracy"],
        "includes_032": "032_ovr_tabm" in keys,
        "weight_032": _weight_of(keys, weights, "032_ovr_tabm"),
        "includes_031": "031_blend_add030" in keys,
        "weight_031": _weight_of(keys, weights, "031_blend_add030"),
        "includes_030": "030_ovr_xgb" in keys,
        "weight_030": _weight_of(keys, weights, "030_ovr_xgb"),
        "includes_029": "029_blend_add028" in keys,
        "weight_029": _weight_of(keys, weights, "029_blend_add028"),
        "includes_028": "028_cat_v3" in keys,
        "weight_028": _weight_of(keys, weights, "028_cat_v3"),
        "weights_json": json.dumps({k: float(w) for k, w in zip(keys, weights)}, ensure_ascii=False) if weights is not None else None,
    }
    row.update({k: v for k, v in ev.items() if k != "balanced_accuracy"})
    if extra:
        row.update(extra)
    blend_rows[(set_name, method)] = row
    if test_p is not None:
        blend_probas[(set_name, method)] = (normalize_rows(oof_p), normalize_rows(test_p))

for set_name, keys in BLEND_SETS.items():
    print("\n===", set_name, keys, "===")

    # 1) Simple average.
    avg_oof, avg_pred = average_proba(keys, oof, pred)
    record_blend(set_name, "avg", keys, avg_oof, avg_pred)

    # 2) Hill climb nonnegative raw probabilities.
    try:
        w, score, hist = hill_climb_weights(y, [oof[k] for k in keys])
        hc_oof = normalize_rows(sum(w[i] * oof[keys[i]] for i in range(len(keys))))
        hc_pred = normalize_rows(sum(w[i] * pred[keys[i]] for i in range(len(keys))))
        record_blend(set_name, "hc_nonnegative_raw", keys, hc_oof, hc_pred, weights=w, extra={"hc_score_internal": score, "hc_n_steps": len(hist)})
    except Exception as e:
        print(f"[WARN] HC failed: {set_name}: {e}")

    # 3) Nelder-Mead softmax weights.
    try:
        w, score, res = nm_softmax_weights(y, [oof[k] for k in keys])
        nm_oof = normalize_rows(sum(w[i] * oof[keys[i]] for i in range(len(keys))))
        nm_pred = normalize_rows(sum(w[i] * pred[keys[i]] for i in range(len(keys))))
        record_blend(set_name, "nm_softmax_raw", keys, nm_oof, nm_pred, weights=w, extra={"nm_score_internal": score, "nm_success": bool(res.success), "nm_message": str(res.message)})
    except Exception as e:
        print(f"[WARN] NM failed: {set_name}: {e}")

    # 4) In-sample meta models. Diagnostic only.
    for mode in ["raw", "rank", "logit", "raw_logit", "raw_rank_logit"]:
        X_meta = build_meta_features(keys, oof, mode)
        X_test_meta = build_meta_features(keys, pred, mode)
        try:
            lr = LogisticRegression(C=0.05, multi_class="multinomial", solver="lbfgs", max_iter=2000, random_state=SEED)
            lr.fit(X_meta, y)
            record_blend(set_name, f"logreg_{mode}", keys, lr.predict_proba(X_meta), lr.predict_proba(X_test_meta), extra={"C": 0.05, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] logreg failed: {set_name} {mode}: {e}")
        try:
            rc = RidgeClassifier(alpha=10.0, random_state=SEED)
            rc.fit(X_meta, y)
            record_blend(set_name, f"ridgeclf_{mode}", keys, softmax_np(rc.decision_function(X_meta)), softmax_np(rc.decision_function(X_test_meta)), extra={"alpha": 10.0, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] ridgeclf failed: {set_name} {mode}: {e}")
        try:
            y_oh = onehot_y(y, n_classes)
            rr = Ridge(alpha=10.0, random_state=SEED)
            rr.fit(X_meta, y_oh)
            record_blend(set_name, f"ridge_reg_{mode}", keys, normalize_rows(np.clip(rr.predict(X_meta), 1e-12, None)), normalize_rows(np.clip(rr.predict(X_test_meta), 1e-12, None)), extra={"alpha": 10.0, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] ridge_reg failed: {set_name} {mode}: {e}")
        try:
            y_oh = onehot_y(y, n_classes)
            oof_cols, test_cols = [], []
            for c in range(n_classes):
                en = ElasticNet(alpha=0.0005, l1_ratio=0.05, random_state=SEED, max_iter=2000)
                en.fit(X_meta, y_oh[:, c])
                oof_cols.append(en.predict(X_meta))
                test_cols.append(en.predict(X_test_meta))
            en_oof = normalize_rows(np.clip(np.vstack(oof_cols).T, 1e-12, None))
            en_pred = normalize_rows(np.clip(np.vstack(test_cols).T, 1e-12, None))
            record_blend(set_name, f"elasticnet_reg_{mode}", keys, en_oof, en_pred, extra={"alpha": 0.0005, "l1_ratio": 0.05, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] elasticnet_reg failed: {set_name} {mode}: {e}")

blend_df = pd.DataFrame(list(blend_rows.values())).sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
display(blend_df.head(200))
blend_df.to_csv(OUTDIR / "blend_diagnostics.csv", index=False)

safe_methods = ["avg", "hc_nonnegative_raw", "nm_softmax_raw"]
safe_blend_df = blend_df[blend_df["method"].isin(safe_methods)].copy().sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
display(safe_blend_df.head(80))
safe_blend_df.to_csv(OUTDIR / "safe_blend_diagnostics.csv", index=False)

focus_032_blend_df = safe_blend_df[safe_blend_df["includes_032"]].copy().sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
display(focus_032_blend_df.head(80))
focus_032_blend_df.to_csv(OUTDIR / "safe_blend_diagnostics_focus_032.csv", index=False)

focus_031_blend_df = safe_blend_df[safe_blend_df["includes_031"]].copy().sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
display(focus_031_blend_df.head(80))
focus_031_blend_df.to_csv(OUTDIR / "safe_blend_diagnostics_focus_031.csv", index=False)



=== A_031_only ['031_blend_add030'] ===

=== B_032_only ['032_ovr_tabm'] ===

=== C_029_only ['029_blend_add028'] ===

=== D_027_only ['027_blend_add026'] ===

=== E_026_only ['026_realmlp_v5'] ===

=== F_028_only ['028_cat_v3'] ===

=== G_030_only ['030_ovr_xgb'] ===

=== H_018_only ['018_xgb_shared003'] ===

=== I_031_032 ['031_blend_add030', '032_ovr_tabm'] ===

=== J_029_032 ['029_blend_add028', '032_ovr_tabm'] ===

=== K_027_032 ['027_blend_add026', '032_ovr_tabm'] ===

=== L_026_032 ['026_realmlp_v5', '032_ovr_tabm'] ===

=== M_028_032 ['028_cat_v3', '032_ovr_tabm'] ===

=== N_030_032 ['030_ovr_xgb', '032_ovr_tabm'] ===

=== O_018_032 ['018_xgb_shared003', '032_ovr_tabm'] ===

=== P_031_029_032 ['031_blend_add030', '029_blend_add028', '032_ovr_tabm'] ===

=== Q_031_028_032 ['031_blend_add030', '028_cat_v3', '032_ovr_tabm'] ===

=== R_031_026_032 ['031_blend_add030', '026_realmlp_v5', '032_ovr_tabm'] ===

=== S_031_030_032 ['031_blend_add030', '030_ovr_xgb', '032_ovr_tabm'] ===



,set_name,method,keys,n_models,balanced_accuracy,includes_032,weight_032,includes_031,weight_031,includes_030,...,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,V_031_027_026_028_030_032,nm_softmax_raw,"031_blend_add030,027_blend_add026,026_realmlp_...",6,0.970040,True,1.303524e-03,True,0.996492,True,...,0.972571,NaN,NaN,0.970040,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
1,R_031_026_032,nm_softmax_raw,"031_blend_add030,026_realmlp_v5,032_ovr_tabm",3,0.970034,True,9.492013e-19,True,0.998729,False,...,0.972571,NaN,NaN,0.970034,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
2,R_031_026_032,hc_nonnegative_raw,"031_blend_add030,026_realmlp_v5,032_ovr_tabm",3,0.970033,True,0.000000e+00,True,1.000000,False,...,0.972571,0.970033,14.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Q_031_028_032,hc_nonnegative_raw,"031_blend_add030,028_cat_v3,032_ovr_tabm",3,0.970033,True,0.000000e+00,True,1.000000,False,...,0.972571,0.970033,13.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,I_031_032,hc_nonnegative_raw,"031_blend_add030,032_ovr_tabm",2,0.970033,True,0.000000e+00,True,1.000000,False,...,0.972571,0.970033,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,L_026_032,elasticnet_reg_raw,"026_realmlp_v5,032_ovr_tabm",2,0.963794,True,NaN,False,NaN,False,...,0.944974,NaN,NaN,NaN,NaN,NaN,NaN,in_sample_meta_screening,0.0005,0.05
196,X_018_019_027_026_028_032,ridge_reg_raw_logit,"018_xgb_shared003,019_blend_best,027_blend_add...",6,0.963784,True,NaN,False,NaN,False,...,0.944393,NaN,NaN,NaN,NaN,NaN,NaN,in_sample_meta_screening,10.0000,NaN
197,X_018_019_027_026_028_032,ridgeclf_raw_logit,"018_xgb_shared003,019_blend_best,027_blend_add...",6,0.963784,True,NaN,False,NaN,False,...,0.944393,NaN,NaN,NaN,NaN,NaN,NaN,in_sample_meta_screening,10.0000,NaN
198,Q_031_028_032,ridgeclf_raw_logit,"031_blend_add030,028_cat_v3,032_ovr_tabm",3,0.963771,True,NaN,True,NaN,False,...,0.944708,NaN,NaN,NaN,NaN,NaN,NaN,in_sample_meta_screening,10.0000,NaN


,set_name,method,keys,n_models,balanced_accuracy,includes_032,weight_032,includes_031,weight_031,includes_030,...,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,V_031_027_026_028_030_032,nm_softmax_raw,"031_blend_add030,027_blend_add026,026_realmlp_...",6,0.970040,True,1.303524e-03,True,0.996492,True,...,0.972571,NaN,NaN,0.970040,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
1,R_031_026_032,nm_softmax_raw,"031_blend_add030,026_realmlp_v5,032_ovr_tabm",3,0.970034,True,9.492013e-19,True,0.998729,False,...,0.972571,NaN,NaN,0.970034,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
2,R_031_026_032,hc_nonnegative_raw,"031_blend_add030,026_realmlp_v5,032_ovr_tabm",3,0.970033,True,0.000000e+00,True,1.000000,False,...,0.972571,0.970033,14.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Q_031_028_032,hc_nonnegative_raw,"031_blend_add030,028_cat_v3,032_ovr_tabm",3,0.970033,True,0.000000e+00,True,1.000000,False,...,0.972571,0.970033,13.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,I_031_032,hc_nonnegative_raw,"031_blend_add030,032_ovr_tabm",2,0.970033,True,0.000000e+00,True,1.000000,False,...,0.972571,0.970033,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73,B_032_only,avg,032_ovr_tabm,1,0.961011,True,NaN,False,NaN,False,...,0.937213,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
74,B_032_only,nm_softmax_raw,032_ovr_tabm,1,0.961011,True,1.000000e+00,False,NaN,False,...,0.937213,NaN,NaN,0.961011,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
75,G_030_only,avg,030_ovr_xgb,1,0.960997,False,NaN,False,NaN,True,...,0.937418,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
76,G_030_only,hc_nonnegative_raw,030_ovr_xgb,1,0.960997,False,NaN,False,NaN,True,...,0.937418,0.960997,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,set_name,method,keys,n_models,balanced_accuracy,includes_032,weight_032,includes_031,weight_031,includes_030,...,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,V_031_027_026_028_030_032,nm_softmax_raw,"031_blend_add030,027_blend_add026,026_realmlp_...",6,0.970040,True,1.303524e-03,True,0.996492,True,...,0.972571,NaN,NaN,0.970040,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
1,R_031_026_032,nm_softmax_raw,"031_blend_add030,026_realmlp_v5,032_ovr_tabm",3,0.970034,True,9.492013e-19,True,0.998729,False,...,0.972571,NaN,NaN,0.970034,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
2,R_031_026_032,hc_nonnegative_raw,"031_blend_add030,026_realmlp_v5,032_ovr_tabm",3,0.970033,True,0.000000e+00,True,1.000000,False,...,0.972571,0.970033,14.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Q_031_028_032,hc_nonnegative_raw,"031_blend_add030,028_cat_v3,032_ovr_tabm",3,0.970033,True,0.000000e+00,True,1.000000,False,...,0.972571,0.970033,13.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,I_031_032,hc_nonnegative_raw,"031_blend_add030,032_ovr_tabm",2,0.970033,True,0.000000e+00,True,1.000000,False,...,0.972571,0.970033,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,S_031_030_032,nm_softmax_raw,"031_blend_add030,030_ovr_xgb,032_ovr_tabm",3,0.970033,True,5.328456e-08,True,0.999982,True,...,0.972571,NaN,NaN,0.970033,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
6,S_031_030_032,hc_nonnegative_raw,"031_blend_add030,030_ovr_xgb,032_ovr_tabm",3,0.970033,True,0.000000e+00,True,1.000000,True,...,0.972571,0.970033,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,I_031_032,nm_softmax_raw,"031_blend_add030,032_ovr_tabm",2,0.970033,True,9.953094e-06,True,0.999990,False,...,0.972571,NaN,NaN,0.970033,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
8,X_018_019_027_026_028_032,hc_nonnegative_raw,"018_xgb_shared003,019_blend_best,027_blend_add...",6,0.970027,True,2.156471e-02,False,NaN,False,...,0.972584,0.970027,15.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Y_031_029_027_026_028_030_032,hc_nonnegative_raw,"031_blend_add030,029_blend_add028,027_blend_ad...",7,0.970024,True,0.000000e+00,True,0.299089,True,...,0.972729,0.970024,14.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,set_name,method,keys,n_models,balanced_accuracy,includes_032,weight_032,includes_031,weight_031,includes_030,...,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,V_031_027_026_028_030_032,nm_softmax_raw,"031_blend_add030,027_blend_add026,026_realmlp_...",6,0.970040,True,1.303524e-03,True,0.996492,True,...,0.972571,NaN,NaN,0.970040,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
1,R_031_026_032,nm_softmax_raw,"031_blend_add030,026_realmlp_v5,032_ovr_tabm",3,0.970034,True,9.492013e-19,True,0.998729,False,...,0.972571,NaN,NaN,0.970034,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
2,R_031_026_032,hc_nonnegative_raw,"031_blend_add030,026_realmlp_v5,032_ovr_tabm",3,0.970033,True,0.000000e+00,True,1.000000,False,...,0.972571,0.970033,14.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Q_031_028_032,hc_nonnegative_raw,"031_blend_add030,028_cat_v3,032_ovr_tabm",3,0.970033,True,0.000000e+00,True,1.000000,False,...,0.972571,0.970033,13.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,I_031_032,hc_nonnegative_raw,"031_blend_add030,032_ovr_tabm",2,0.970033,True,0.000000e+00,True,1.000000,False,...,0.972571,0.970033,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,S_031_030_032,nm_softmax_raw,"031_blend_add030,030_ovr_xgb,032_ovr_tabm",3,0.970033,True,5.328456e-08,True,0.999982,True,...,0.972571,NaN,NaN,0.970033,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
6,S_031_030_032,hc_nonnegative_raw,"031_blend_add030,030_ovr_xgb,032_ovr_tabm",3,0.970033,True,0.000000e+00,True,1.000000,True,...,0.972571,0.970033,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,I_031_032,nm_softmax_raw,"031_blend_add030,032_ovr_tabm",2,0.970033,True,9.953094e-06,True,0.999990,False,...,0.972571,NaN,NaN,0.970033,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
8,A_031_only,avg,031_blend_add030,1,0.970033,False,NaN,True,NaN,False,...,0.972571,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,A_031_only,nm_softmax_raw,031_blend_add030,1,0.970033,False,NaN,True,1.000000,False,...,0.972571,NaN,NaN,0.970033,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN


In [6]:
# ============================================================
# 5. Save best safe blend
# ============================================================

safe_blend_df = safe_blend_df.sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
best_row = safe_blend_df.iloc[0].to_dict()
best_set_name = best_row["set_name"]
best_method = best_row["method"]
best_keys = best_row["keys"].split(",")

print("Best safe blend:")
print(json.dumps(best_row, indent=2, ensure_ascii=False))

if best_method == "avg":
    best_oof_proba, best_pred_proba = average_proba(best_keys, oof, pred)
else:
    weights_json = json.loads(best_row["weights_json"])
    w = np.array([weights_json[k] for k in best_keys], dtype=float)
    best_oof_proba = normalize_rows(sum(w[i] * oof[best_keys[i]] for i in range(len(best_keys))))
    best_pred_proba = normalize_rows(sum(w[i] * pred[best_keys[i]] for i in range(len(best_keys))))

validate_proba("best_oof_proba", best_oof_proba, len(train), n_classes)
validate_proba("best_pred_proba", best_pred_proba, len(test), n_classes)

best_oof_score = balanced_accuracy_score(y, best_oof_proba.argmax(axis=1))
print("best_oof_score:", best_oof_score)

best_oof_npy = OUTDIR / f"oof_{EXP_ID}_best_blend_proba.npy"
best_pred_npy = OUTDIR / f"pred_{EXP_ID}_best_blend_proba.npy"
np.save(best_oof_npy, best_oof_proba.astype(np.float32))
np.save(best_pred_npy, best_pred_proba.astype(np.float32))

best_labels = le.inverse_transform(best_pred_proba.argmax(axis=1))
submission = pd.DataFrame({ID_COL: test[ID_COL].values, TARGET_COL: best_labels})
assert submission.shape[0] == sample.shape[0]
assert submission.columns.tolist() == sample.columns.tolist()
assert submission[ID_COL].equals(sample[ID_COL])

best_submission_path = OUTDIR / f"submission_{EXP_ID}_best_blend.csv"
submission.to_csv(best_submission_path, index=False)

best_info = {
    "exp_id": EXP_ID,
    "best_set_name": best_set_name,
    "best_method": best_method,
    "best_keys": best_keys,
    "cv_score": float(best_oof_score),
    "includes_032": "032_ovr_tabm" in best_keys,
    "weight_032": best_row.get("weight_032"),
    "includes_031": "031_blend_add030" in best_keys,
    "weight_031": best_row.get("weight_031"),
    "includes_030": "030_ovr_xgb" in best_keys,
    "weight_030": best_row.get("weight_030"),
    "includes_029": "029_blend_add028" in best_keys,
    "weight_029": best_row.get("weight_029"),
    "includes_028": "028_cat_v3" in best_keys,
    "weight_028": best_row.get("weight_028"),
    "weights_json": best_row.get("weights_json"),
    "oof_path": str(best_oof_npy),
    "pred_path": str(best_pred_npy),
    "submission_path": str(best_submission_path),
    "row": best_row,
    "note": "Best safe blend excludes in-sample meta screening methods.",
}
save_json(best_info, OUTDIR / "best_blend_info.json")

saved_submission_df = pd.DataFrame([{
    "set_name": best_set_name,
    "method": best_method,
    "balanced_accuracy": float(best_oof_score),
    "includes_032": "032_ovr_tabm" in best_keys,
    "weight_032": best_row.get("weight_032"),
    "includes_031": "031_blend_add030" in best_keys,
    "weight_031": best_row.get("weight_031"),
    "submission": best_submission_path.name,
    "oof_proba": best_oof_npy.name,
    "pred_proba": best_pred_npy.name,
}])
display(saved_submission_df)
saved_submission_df.to_csv(OUTDIR / "saved_submission_candidates.csv", index=False)


Best safe blend:
{
  "set_name": "V_031_027_026_028_030_032",
  "method": "nm_softmax_raw",
  "keys": "031_blend_add030,027_blend_add026,026_realmlp_v5,028_cat_v3,030_ovr_xgb,032_ovr_tabm",
  "n_models": 6,
  "balanced_accuracy": 0.9700400336552478,
  "includes_032": true,
  "weight_032": 0.001303523571311227,
  "includes_031": true,
  "weight_031": 0.996491852378833,
  "includes_030": true,
  "weight_030": 1.487009143685007e-12,
  "includes_029": false,
  "weight_029": NaN,
  "includes_028": true,
  "weight_028": 7.505743518983776e-05,
  "weights_json": "{\"031_blend_add030\": 0.996491852378833, \"027_blend_add026\": 0.0010151681622706588, \"026_realmlp_v5\": 0.0011143984509084026, \"028_cat_v3\": 7.505743518983776e-05, \"030_ovr_xgb\": 1.487009143685007e-12, \"032_ovr_tabm\": 0.001303523571311227}",
  "recall_GALAXY": 0.9617754583024266,
  "recall_QSO": 0.975773200276585,
  "recall_STAR": 0.9725714423867318,
  "hc_score_internal": NaN,
  "hc_n_steps": NaN,
  "nm_score_internal": 0.97

,set_name,method,balanced_accuracy,includes_032,weight_032,includes_031,weight_031,submission,oof_proba,pred_proba
0,V_031_027_026_028_030_032,nm_softmax_raw,0.97004,True,0.001304,True,0.996492,submission_exp_20260609_033_blend_add032_tabm_...,oof_exp_20260609_033_blend_add032_tabm_check_b...,pred_exp_20260609_033_blend_add032_tabm_check_...


In [7]:
# ============================================================
# 6. Save summary / memo
# ============================================================

reference_scores = {
    "031_cv": 0.9700333620193362,
    "031_public_lb": 0.97043,
    "029_cv": 0.9700023026377228,
    "029_public_lb": 0.970036,
    "027_cv": 0.9695425457477898,
    "027_public_lb": 0.96975,
    "026_cv": 0.9690389777981231,
    "026_public_lb": 0.96979,
    "028_cv": 0.9688146470512534,
    "028_public_lb": 0.96969,
    "019_cv": 0.968872315017554,
    "019_public_lb": 0.97000,
    "018_cv": 0.965207986418628,
    "018_public_lb": 0.96578,
    "030_cv": 0.9609971296999271,
    "030_public_lb": 0.96118,
    "032_raw_cv": 0.9610105651284856,
    "032_raw_public_lb": 0.96106,
    "032_tuned_cv": 0.9686168281485955,
    "032_biased_public_lb": 0.96895,
}

# Important derived checks for judging 032.
best_032_safe = focus_032_blend_df.iloc[0].to_dict() if len(focus_032_blend_df) else None
best_safe_weight_032 = best_info.get("weight_032")

judgement = {
    "reference_scores": reference_scores,
    "individual_best": {
        "key": individual_df.iloc[0]["key"],
        "cv": float(individual_df.iloc[0]["recomputed_oof_cv"]),
        "public_lb": float(individual_df.iloc[0]["public_lb"]),
    },
    "best_safe_blend": best_info,
    "best_safe_blend_with_032": best_032_safe,
    "main_questions": {
        "does_032_add_to_031": "Check I/P/Q/R/S/V/Y safe methods and 032 weight.",
        "does_032_add_to_029_or_core": "Check J/T/U/W/X/Y safe methods and 032 weight.",
        "does_032_add_to_028_030_family": "Check M/N/S/U/W safe methods.",
        "should_deepen_032_next": "Only if 032 receives meaningful non-zero safe blend weight and improves CV.",
        "should_replace_submission_slot": "Only if best safe blend beats 031 CV and Public LB confirms. Otherwise keep 031 as current best.",
    },
    "automatic_view": {
        "best_safe_includes_032": bool(best_info.get("includes_032")),
        "best_safe_weight_032": best_safe_weight_032,
        "best_032_safe_cv": None if best_032_safe is None else float(best_032_safe["balanced_accuracy"]),
        "best_032_safe_method": None if best_032_safe is None else best_032_safe["method"],
        "best_032_safe_set": None if best_032_safe is None else best_032_safe["set_name"],
        "032_keep_rule": (
            "Keep if 032 weight is non-trivial and best safe CV improves over 031. "
            "If weight is near zero or CV does not improve, keep only as optional material and proceed to 034."
        ),
    },
}
save_json(judgement, OUTDIR / "role_judgement.json")

summary = {
    "competition": COMPETITION,
    "exp_id": EXP_ID,
    "status": "completed",
    "purpose": "Blend/correlation check after adding 032 OVR TabM as-is to 031/029/027/026/028/030/018 candidate pool",
    "npy_dir": str(NPY_DIR),
    "model_keys": model_keys,
    "class_names": class_names,
    "resolved_specs": resolved_specs,
    "individual_scores": individual_df.to_dict(orient="records"),
    "pairwise_oof_correlation": pairwise_df.to_dict(orient="records"),
    "pairwise_oof_correlation_focus_032": focus_032_df.to_dict(orient="records"),
    "pairwise_oof_correlation_focus_031": focus_031_df.to_dict(orient="records"),
    "pairwise_oof_correlation_focus_029": focus_029_df.to_dict(orient="records"),
    "blend_diagnostics": blend_df.to_dict(orient="records"),
    "safe_blend_diagnostics": safe_blend_df.to_dict(orient="records"),
    "safe_blend_diagnostics_focus_032": focus_032_blend_df.to_dict(orient="records"),
    "safe_blend_diagnostics_focus_031": focus_031_blend_df.to_dict(orient="records"),
    "best_blend_info": best_info,
    "saved_submission_candidates": saved_submission_df.to_dict(orient="records"),
    "role_judgement": judgement,
}
save_json(summary, OUTDIR / "blend_add032_tabm_summary.json")

memo = {
    "experiment": {
        "id": EXP_ID,
        "title": "Blend check after adding 032 OVR TabM",
        "competition": COMPETITION,
        "status": "completed",
        "metric": "balanced_accuracy_score",
        "created_at": "2026-06-09",
    },
    "objective": {
        "summary": (
            "Load OOF/pred probabilities from npy-files, add 032 OVR TabM as-is, "
            "and decide whether this TabM material adds complementary signal to 031 or to the 027/026/028/030 core."
        ),
        "success_criteria": [
            "load 015/017/018/019/020/024/026/027/028/029/030/031/032 OOF and pred npy files",
            "recompute individual balanced accuracy",
            "compute pairwise OOF correlations and focus table for 032",
            "evaluate add032 blend sets",
            "separate safe blends from in-sample meta screening",
            "save best safe blend OOF/pred npy",
            "save best safe blend submission",
        ],
    },
    "inputs": {"npy_dir": str(NPY_DIR), "models": resolved_specs, "blend_sets": BLEND_SETS},
    "outputs": {
        "individual_scores": "individual_scores.csv",
        "pairwise_oof_correlation": "pairwise_oof_correlation.csv",
        "pairwise_oof_correlation_focus_032": "pairwise_oof_correlation_focus_032.csv",
        "pairwise_oof_correlation_focus_031": "pairwise_oof_correlation_focus_031.csv",
        "pairwise_oof_correlation_focus_029": "pairwise_oof_correlation_focus_029.csv",
        "blend_diagnostics": "blend_diagnostics.csv",
        "safe_blend_diagnostics": "safe_blend_diagnostics.csv",
        "safe_blend_diagnostics_focus_032": "safe_blend_diagnostics_focus_032.csv",
        "safe_blend_diagnostics_focus_031": "safe_blend_diagnostics_focus_031.csv",
        "best_blend_info": "best_blend_info.json",
        "best_oof_proba": best_oof_npy.name,
        "best_pred_proba": best_pred_npy.name,
        "best_submission": best_submission_path.name,
        "summary": "blend_add032_tabm_summary.json",
    },
    "judgement": {
        "status": "completed_pending_manual_review",
        "initial_view": (
            "032 biased submission is usable but not a slot update. "
            "Promote 032 only if it improves safe blend CV or receives meaningful non-zero safe blend weight."
        ),
        "automatic_view": judgement["automatic_view"],
        "next_action": [
            "Review individual_scores.csv and confirm whether 032 npy recomputed CV matches raw CV",
            "Review pairwise_oof_correlation_focus_032.csv, especially 032 vs 031/029/026/028/030",
            "Review safe_blend_diagnostics_focus_032.csv",
            "If best safe blend improves over 031, submit best_submission",
            "If 032 weight is near zero or no CV improvement, proceed to 034 GPU logistic regression stacker",
            "Do not treat in-sample LogReg/Ridge/ElasticNet rows as final candidates",
        ],
    },
}

memo_path = OUTDIR / "memo.yml"
if yaml is not None:
    with open(memo_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(memo, f, allow_unicode=True, sort_keys=False)
else:
    with open(memo_path, "w", encoding="utf-8") as f:
        f.write(json.dumps(memo, indent=2, ensure_ascii=False, default=str))

print("Saved files:")
for p in sorted(OUTDIR.glob("*")):
    print(" -", p.name)


Saved files:
 - best_blend_info.json
 - blend_add032_tabm_summary.json
 - blend_diagnostics.csv
 - individual_scores.csv
 - memo.yml
 - oof_exp_20260609_033_blend_add032_tabm_check_best_blend_proba.npy
 - pairwise_oof_correlation.csv
 - pairwise_oof_correlation_focus_029.csv
 - pairwise_oof_correlation_focus_031.csv
 - pairwise_oof_correlation_focus_032.csv
 - pred_exp_20260609_033_blend_add032_tabm_check_best_blend_proba.npy
 - role_judgement.json
 - safe_blend_diagnostics.csv
 - safe_blend_diagnostics_focus_031.csv
 - safe_blend_diagnostics_focus_032.csv
 - saved_submission_candidates.csv
 - submission_exp_20260609_033_blend_add032_tabm_check_best_blend.csv
